## Training
Prepapre dataset with the prepare_dataset notebook, before running this one.

In [ ]:
from torch.utils.data import DataLoader
# from tqdm import tqdm
from time import time, sleep
import numpy as np
import os
import warnings; warnings.filterwarnings("ignore")
from utils import *
  
# DEV = torch.device(CPU) # for debugging, use cpu
os.makedirs(f"{SAVE_DIR}", exist_ok=True)
print(f'DEV: {DEV}, has_screen: {LOCAL}, job id: {JOBID}')

# copy the python training to the directory (for cluster) (for local, it fails silently)
os.system(f"cp train.py {SAVE_DIR}/train.py")
os.system(f"cp utils.py {SAVE_DIR}/utils.py")

In [ ]:
EPOCHS = 12 if LOCAL else 50 # number of epochs 100 <-
# BATCH_SIZE = 128 # 128 
BATCH_SIZE = 64 if LOCAL else 512 # 64 <-
PTS_PER_SAMPLE = 10 if LOCAL else 50 # number of points to pick for each training sample in the ds, must be a perfect divider of total number of pts per time instant
print(f'EPOCHS: {EPOCHS}, BATCH_SIZE: {BATCH_SIZE}, PTS_PER_SAMPLE: {PTS_PER_SAMPLE}')

# LOAD_PRETRAINED = f'{BEST_MODEL_DIR}/best_{FX}.pth' if LOCAL else None # pretrained model
LOAD_PRETRAINED = None # Set it to None if you don't want to load pretrained model
# LOAD_PRETRAINED = 'data/3067354/best_Fx.pth'

In [ ]:
# schedulers
from numpy import concatenate as cat, linspace as linsp, logspace as logsp
### learning rate
# LR = 3e-4*logsp(0, -2, EPOCHS) 
# LR = 3e-3*np.ones(EPOCHS) 
# LR = 3e-3*logsp(0, -2, EPOCHS)  
# LR = 1e-3*np.ones(EPOCHS) if LOCAL else 1e-3*logsp(0, -2, EPOCHS) # <-
LR = 5e-3*logsp(0, -2, EPOCHS) # <- [good]
# LR = 5e-4*np.ones(EPOCHS)
# LR = 2e-3*np.ones(EPOCHS)

### loss ratios 
## Fx
R_FX = np.ones(EPOCHS) 
# R_FX = 1.0 * logsp(0, -4, EPOCHS) # <- for Iy estimate
# R_FX = np.zeros(EPOCHS) 
# R_FX = 1e-5*np.ones(EPOCHS) 

## Iy
# R_IY  = 0.1 * logsp(0, -4, EPOCHS) 
# R_IY  = 0.1 * np.ones(EPOCHS) 
R_IY = np.zeros(EPOCHS) # <- best for Fx

## Br Bz
# R_B = 0.1 * np.ones(EPOCHS) 
# R_B = 1.0 * np.ones(EPOCHS) 
# R_B = 0.1 * logsp(0, -4, EPOCHS) 
R_B = np.zeros(EPOCHS) # <- [best, use grad of Fx to get Br Bz]

## Br Bz from derivative, PINN
R_BD = 1e-1 * np.ones(EPOCHS) # <-
# R_BD = np.ones(EPOCHS) 
# R_BD = 1e-5*np.ones(EPOCHS) 
# R_BD = np.zeros(EPOCHS)  

## Iy from second derivative, PINN
R_IYD = 1e-2 * np.ones(EPOCHS)
# R_IYD = np.zeros(EPOCHS)

ACTIVE_LOSSES = [np.sum(l_profile) > 0 for l_profile in [R_FX, R_IY, R_B, R_BD, R_IYD]]

In [ ]:
#checks
if LOAD_PRETRAINED is not None: assert os.path.exists(LOAD_PRETRAINED), "Pretrained model does not exist"
assert os.path.exists(TRAIN_DS_PATH), "Training dataset does not exist"
assert os.path.exists(EVAL_DS_PATH), "Evaluation dataset does not exist"
assert os.path.exists(SAVE_DIR), "Save directory does not exist"
assert len(LR) == EPOCHS, "Learning rate array length does not match epochs"
assert len(R_FX) == EPOCHS, "R_FX loss ratio array length does not match epochs"
assert len(R_IY) == EPOCHS, "R_IY loss ratio array length does not match epochs"
assert len(R_B) == EPOCHS, "R_B loss ratio array length does not match epochs"
assert len(ACTIVE_LOSSES) == len(LOSS_NAMES), "ACTIVE_LOSSES length does not match LOSS_NAMES"

In [ ]:
# # test net I/O
test_network_io()

In [ ]:
# plot schedulers
n_ticks = 11
plt.figure(figsize=(12, 12))
titles = ['Learning Rate', 'Fx Loss Ratio', 'Iy Loss Ratio', 'Br Bz Loss Ratio', 'Br Bz from Derivative Loss Ratio', 'Iy from 2nd Derivative Loss Ratio']
pi = 2 # plot index
plt.subplot(len(ACTIVE_LOSSES)+1, 1, 1), plt.plot(LR), plt.yticks(np.linspace(0, np.max(LR), n_ticks)), plt.title(titles[0])
active_idxs = [i for i, active in enumerate(ACTIVE_LOSSES) if active]
for i, r in enumerate([LR, R_FX, R_IY, R_B, R_BD, R_IYD]):
    if i==0 or not ACTIVE_LOSSES[i-1]: continue
    plt.subplot(len(ACTIVE_LOSSES)+1, 1, pi), plt.plot(r), plt.yticks(np.linspace(0, np.max(r), n_ticks)), plt.title(titles[i])
    pi += 1
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(f"{SAVE_DIR}/schedulers.png")

In [ ]:
# create training and evaluation datasets
train_ds = LiuqeDataset(TRAIN_DS_PATH, pts_per_sample=PTS_PER_SAMPLE)
val_ds = LiuqeDataset(EVAL_DS_PATH, pts_per_sample=PTS_PER_SAMPLE)
test_dataset(val_ds)
#save the mean_std of the dataset int the save directory
np.savez(f"{SAVE_DIR}/x_mean_std.npz", x_mean_std=train_ds.x_mean_std.cpu().numpy())

## Training

In [ ]:
def train():
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True) # initialize DataLoader
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False) # initialize DataLoader
    # network
    model = liuqeML(train_ds.x_mean_std)
    opt = torch.optim.Adam(model.parameters(), lr=LR[0])# adam
    # opt = torch.optim.AdamW(model.parameters(), lr=LR[0], weight_decay=1e-4) # adamw
    # opt = torch.optim.Muon(model.parameters(), lr=LR[0]) # Muon optimizer

    if LOAD_PRETRAINED is not None: # load pretrained model
        model.load_state_dict(torch.load(LOAD_PRETRAINED, map_location=torch.device(CPU)), strict=STRICT_LOAD) # load pretrained model
        print(f"Pretrained model loaded: {LOAD_PRETRAINED}")

    model.to(DEV) # move model to DEV
    loss_fn = F.mse_loss # Mean Squared Error
    # loss_fn = F.l1_loss # Mean Absolute Error
    tlog, elog = np.inf*np.ones((EPOCHS, len(train_dl), len(LOSS_NAMES))), np.inf*np.ones((EPOCHS, len(val_dl), len(LOSS_NAMES))) # init log
    start_time = time() # start time
    print("Ep:   || fx    | iy  |  b  |  bd | iyd || r:fx| r:iy| r:b | r:bd|r:iyd| LR  || t  | eta| improved    ")
    for ep in range(EPOCHS): # epochs
        epoch_time = time()
        for pg in opt.param_groups: pg['lr'] = LR[ep] # update learning rate
        model.train()
        for ib, batch in enumerate(train_dl):
            if train_ds.on_dev: x, p, fx, iy, br, bz = batch # unpack batch
            else: x, p, fx, iy, br, bz = map(lambda t: t.to(DEV), batch) # move to DEV
            r = p.clone()[:,:,0] # r coordinates
            p.requires_grad = True # enable gradients for p

            opt.zero_grad() # zero gradients
            fxp, brzp, iyp = model(x, p) # forward pass
            brp, bzp = brzp[:,:,0], brzp[:,:,1] # unpack into br and bz

            if R_BD[ep] > 0 or R_IYD[ep] > 0: # calculate only if needed
                # calculate first derivatives of Fx wrt r and z, for Br Bz
                dfx_drz, = torch.autograd.grad(outputs=fxp.sum(), inputs=p, create_graph=True)
                dfx_dr, dfx_dz = dfx_drz[:,:,0], dfx_drz[:,:,1] # derivative of Fx wrt r and z
                brpd = -dfx_dz / (2 * torch.pi * r) # Br from derivative
                bzpd = dfx_dr / (2 * torch.pi * r) # Bz from derivative
                if R_IYD[ep] > 0:
                    # calculate second derivatives of Fx wrt r and z for Iy
                    d2fx_dr2 = torch.autograd.grad(outputs=dfx_dr.sum(), inputs=p, create_graph=True)[0][:,:,0]
                    d2fx_dz2 = torch.autograd.grad(outputs=dfx_dz.sum(), inputs=p, create_graph=True)[0][:,:,1]
                    iydp = - (1/(2*π*r) * d2fx_dz2 + 1/(2*π*r) * d2fx_dr2 - 1/(2*π*r**2) * dfx_dr)
                else: iydp = iy # dummy
            else: brpd, bzpd, iydp = br, bz, iy # dummy

            # losses
            l_fx = loss_fn(fx, fxp) # fx loss
            l_iy = loss_fn(iy, iyp) # iy loss
            l_brz = loss_fn(br, brp) + loss_fn(bz, bzp) # br bz loss
            l_brzd = loss_fn(br, brpd) + loss_fn(bz, bzpd) # br bz loss from derivative
            l_iyd = loss_fn(iy, iydp) # iy loss from second derivative

            loss = R_FX[ep]*l_fx + R_IY[ep]*l_iy + R_B[ep]*l_brz + R_BD[ep]*l_brzd + R_IYD[ep]*l_iyd # total loss
            loss.backward() # backprop

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) # gradient clipping
    
            opt.step() # update weights

            tlog[ep, ib] = [l.item() for l in (l_fx, l_iy, l_brz, l_brzd, l_iyd)] # save batch losses

        model.eval() # evaluation mode
        with torch.no_grad(): # Disable gradients for most of the validation
            for ib, batch in enumerate(val_dl):
                if val_ds.on_dev: x, p, fx, iy, br, bz = batch # unpack batch
                else: x, p, fx, iy, br, bz = map(lambda t: t.to(DEV), batch) # move to DEV
                with torch.enable_grad():
                    p.requires_grad = True # enable gradients for p
                    fxp, brzp, iyp = model(x, p) # forward pass
                    brp, bzp = brzp[:,:,0], brzp[:,:,1] # unpack outputs
                    r = p.clone()[:,:,0] # r coordinates    
                    if R_BD[ep] > 0: dfx_drz, = torch.autograd.grad(outputs=fxp.sum(), inputs=p, create_graph=True)
                    else: dfx_drz = torch.zeros_like(p)
                    dfx_dr, dfx_dz = dfx_drz[:,:,0], dfx_drz[:,:,1] # derivative of Fx wrt r and z
                    if R_BD[ep] > 0: 
                        # calculate second derivatives of Fx wrt r and z for Iy
                        d2fx_dr2 = torch.autograd.grad(outputs=dfx_dr.sum(), inputs=p, create_graph=True)[0][:,:,0]
                        d2fx_dz2 = torch.autograd.grad(outputs=dfx_dz.sum(), inputs=p, create_graph=True)[0][:,:,1]
                    else: d2fx_d2rz = torch.zeros_like(p)
                brpd = -dfx_dz / (2 * torch.pi * r) # Br from derivative
                bzpd = dfx_dr / (2 * torch.pi * r) # Bz from derivative
                iydp = - (1/(2*π*r) * d2fx_dz2 + 1/(2*π*r) * d2fx_dr2 - 1/(2*π*r**2) * dfx_dr) # Iy from second derivative
                # losses
                l_fx = loss_fn(fx, fxp) # fx loss
                l_iy = loss_fn(iy, iyp) # iy loss
                l_brz = loss_fn(br, brp) + loss_fn(bz, bzp) # br bz loss
                l_brzd = loss_fn(br, brpd) + loss_fn(bz, bzpd) # br bz loss from derivative
                l_iyd = loss_fn(iy, iydp) # iy loss from second derivative
                elog[ep, ib] = [l.item() for l in (l_fx, l_iy, l_brz, l_brzd, l_iyd)] # save batch losses

        # save model if improved
        endp = ""
        epoch_losses = np.mean(elog[ep,:,:], axis=0) # epoch losses: mean across batchess
        best_losses = np.min(np.mean(elog[:ep,:,:], axis=1), axis=0) if ep > 0 else epoch_losses # best losses: min across epochs
        for el, bl, n in zip(epoch_losses, best_losses, LOSS_NAMES):
            if el <= bl: torch.save(model.state_dict(), model_path(n)); endp+=f"*{n}"
        # print progress
        print(f"{ep+1:03d}/{EPOCHS:02d}||" + f"{epoch_losses[0]:.1e}|" +
            '|'.join([f"{v:.0e}" for v in epoch_losses[1:]]) + "||" +
            f"{R_FX[ep]:.0e}|{R_IY[ep]:.0e}|{R_B[ep]:.0e}|{R_BD[ep]:.0e}|{R_IYD[ep]:.0e}|{LR[ep]:.0e}||" +
            f"{int(time()-epoch_time):03d}s|{int((time()-start_time)*(EPOCHS-ep)/(ep+1)/(1 if LOCAL else 60)):03d}{'s' if LOCAL else 'm'}|", end=endp+'\n')
        # if ep >= 10 and best_losses[2] > 0.1: return False

    tlosses, elosses = np.mean(tlog, axis=1), np.mean(elog, axis=1) # losses across epochs
    print(f"Training time: {(time()-start_time)/60:.0f}mins")
    print("Best MSE: " + ", ".join([f"{n} {bl:.3e} (ep {be})" for n, bl, be in zip(LOSS_NAMES, np.min(elosses, axis=0), np.argmin(elosses, axis=0))]))
    print("Estimated MAE: " + ", ".join([f"{n} {np.sqrt(np.min(np.mean(elog[:,:,i], axis=1))):.3e}" for i, n in enumerate(LOSS_NAMES)]))
    np.save(f"{SAVE_DIR}/train_losses.npy", tlosses) # save training losses
    np.save(f"{SAVE_DIR}/eval_losses.npy", elosses) # save evaluation losses
    return True

# train the model (multiple attempts)
for i in range(20): 
    print('Starting training...')
    success = train()
    if success: break
    else: print(f"Convergence failed, retrying... {i+1}/20")
# if not success: delete the files and exit:
if not success: os.system(f"rm -rf {SAVE_DIR}")
assert success, "Training failed, no model saved"

In [ ]:
# plot losses
print("Plotting losses...")
# fig, ax = plt.subplots(2, len(LOSS_NAMES), figsize=(4*len(LOSS_NAMES), 8))
fig = plt.figure(figsize=(4*len(LOSS_NAMES), 8))
# Load losses
train_loss = np.load(f"{SAVE_DIR}/train_losses.npy")
eval_loss = np.load(f"{SAVE_DIR}/eval_losses.npy")
n_active_losses = sum(ACTIVE_LOSSES)
active_names = [name for i, name in enumerate(LOSS_NAMES) if ACTIVE_LOSSES[i]]
active_indices = [i for i, active in enumerate(ACTIVE_LOSSES) if active]
for i, name in enumerate(active_names):
    # Linear scale
    plt.subplot(2, n_active_losses, i+1)
    plt.plot(train_loss[:,active_indices[i]], label='train')
    plt.plot(eval_loss[:,active_indices[i]], label='eval')
    plt.title(f"{name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    # Log scale
    plt.subplot(2, n_active_losses, i+1 + n_active_losses)
    plt.plot(train_loss[:,active_indices[i]], label='train')
    plt.plot(eval_loss[:,active_indices[i]], label='eval')
    plt.yscale('log')
    plt.title(f"{name} (log)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(f"{SAVE_DIR}/losses.png")

In [ ]:
def plot_network_outputs(ds:LiuqeDataset, model:liuqeML, title="test", save_dir=SAVE_DIR, nplt=2 if LOCAL else 50, use_db=True, use_diy=True):
    assert ds.fg is not None, "Dataset does not have full grid data (fg)"
    if use_diy: assert use_db, "use_diy requires use_db to be True"
    _ds_pps = ds.pps # tmp workaround for proper plotting
    ds.pps = ds.tnp # make ds to sample every point for every timestamp
    main_col = 'w' if DARK_THEME else 'k'
    if not use_db: model.eval()
    model.to(CPU)
    os.makedirs(f"{save_dir}/imgs", exist_ok=True)
    for pi, i in enumerate(np.random.randint(0, len(ds), nplt)):  
        on_grid = pi >= (nplt // 2) # on grid or not
        if not on_grid: continue # # TODO remove this
        fs = (20, 8) if on_grid else (15, 8) # figure size
        plt.figure(figsize=fs, facecolor='white' if DARK_THEME else 'black')
        x, p, fx, iy, br, bz = ds[i]
        x, p, fx, iy, br, bz = map(lambda x: x.to(CPU), [x, p, fx, iy, br, bz])
        if on_grid: 
            μ0 = 4e-7 * np.pi  # Permeability of free space
            p = torch.stack([torch.tensor(RRD.reshape(-1), dtype=torch.float32), torch.tensor(ZZD.reshape(-1), dtype=torch.float32)], dim=1)  # shape (n, 2)
            fx, iy, br, bz = [to_tensor(v.reshape(-1)) for v in ds.fg[i]] # reshape to (n,)
            iy = iy*μ0/GRA # workaround, on grid points dont have the conversion applied
        if use_db: r = p.clone()[:,0]; p.requires_grad = True # need r for PINN
        reverse_cmap = iy.mean() < 0  # reverse cmap if current density is negative
        fxp, brzp, iyp = model(x.unsqueeze(0), p.unsqueeze(0))
        if use_db:
            dfx_drz = torch.autograd.grad(outputs=fxp.sum(), inputs=p, create_graph=True)[0].unsqueeze(0) # compute gradients wrt p
            π = torch.pi
            dfx_dr, dfx_dz = dfx_drz[:,:,0], dfx_drz[:,:,1]  # shape (1, n)
            brp = -dfx_dz / (2*π*r) # Br from derivative
            bzp = dfx_dr / (2*π*r)  # Bz from derivative
            if use_diy: # calculate second derivative
                d2fx_dr2 = torch.autograd.grad(outputs=dfx_dr.sum(), inputs=p, create_graph=True)[0][:,0].unsqueeze(0)
                d2fx_dz2 = torch.autograd.grad(outputs=dfx_dz.sum(), inputs=p, create_graph=True)[0][:,1].unsqueeze(0)
                iyp = - (1/(2*π*r) * d2fx_dz2 + 1/(2*π*r) * d2fx_dr2 - 1/(2*π*r**2) * dfx_dr)

        else: brp, bzp = brzp[:,:,0], brzp[:,:,1]  # split Br, Bz
        if True:
            fxp, iyp, brp, bzp = map(lambda t: t.squeeze(0), [fxp, iyp, brp, bzp]) # remove batch dimension
            p, fx, iy, br, bz, fxp, iyp, brp, bzp = map(lambda t: t.detach().cpu().numpy(), [p, fx, iy, br, bz, fxp, iyp, brp, bzp]) # move to CPU and numpy
            iv = inpoly(p) # inside vessel indices
            # mfx, Mfx = np.min([fx[iv], fxp[iv]]), np.max([fx[iv], fxp[iv]]) # both maps
            # miy, Miy = np.min([iy[iv], iyp[iv]]), np.max([iy[iv], iyp[iv]])
            # mbr, Mbr = np.min([br[iv], brp[iv]]), np.max([br[iv], brp[iv]])
            # mbz, Mbz = np.min([bz[iv], bzp[iv]]), np.max([bz[iv], bzp[iv]])
            mfx, Mfx = np.min(fx[iv]), np.max(fx[iv]) # only actual
            miy, Miy = np.min(iy[iv]), np.max(iy[iv])
            mbr, Mbr = np.min(br[iv]), np.max(br[iv])
            mbz, Mbz = np.min(bz[iv]), np.max(bz[iv])
            # mfx, Mfx = np.min(fxp[iv]), np.max(fxp[iv]) # only predicted
            # miy, Miy = np.min(iyp[iv]), np.max(iyp[iv])
            # mbr, Mbr = np.min(brp[iv]), np.max(brp[iv])
            # mbz, Mbz = np.min(bzp[iv]), np.max(bzp[iv]) 
            lw, col, ms, cm = 1.5, 'gray', 4, plt.rcParams['image.cmap'] + ('_r' if reverse_cmap else '') # line width, color, marker size, colormap
            limits = [(mfx, Mfx), (miy, Miy), (mbr, Mbr), (mbz, Mbz)] # variable limits
            vals = [(fx, fxp), (iy, iyp), (br, brp), (bz, bzp)]
            names = ["Fx", "Iyd" if use_diy else "Iy", "Brd" if use_db else "Br", "Bzd" if use_db else "Bz"]
            def prep_plot(ax):
                ax.grid(False)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_aspect("equal")
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.spines['left'].set_visible(False)
                ax.spines['bottom'].set_visible(False)
                plot_vessel(ax) # plot vessel
            nraws = 2
            ncols = 8 if on_grid else 6 # number of columns
            α_sep = 0.5
            for j, ((v, vp), l, n) in enumerate(zip(vals, limits, names)):
                plt.subplot(nraws, ncols, j*(ncols//2)+1), plt.title(f"{n} Actual")
                plt.scatter(p[iv,0], p[iv,1], c=v[iv], s=ms, vmin=l[0], vmax=l[1], cmap=cm)
                prep_plot(plt.gca()), plt.colorbar(); reverse_cmap and plt.gcf().axes and plt.gcf().axes[-1].invert_yaxis(); 
                plt.subplot(nraws, ncols, j*(ncols//2) + 2); plt.title(f"{n} Predicted")
                plt.scatter(p[iv,0], p[iv,1], c=vp[iv], s=ms, vmin=l[0], vmax=l[1], cmap=cm)
                prep_plot(plt.gca()), plt.colorbar(); reverse_cmap and plt.gcf().axes and plt.gcf().axes[-1].invert_yaxis();
                plt.subplot(nraws, ncols, j*(ncols//2) + ncols//2), plt.title(f"{n} Error %")
                ep = 100*np.abs(v[iv]-vp[iv])/(l[1]-l[0]) # error in %
                # plt.scatter(p[iv,0], p[iv,1], c=ep, s=ms, vmin=0, vmax=np.max(ep)) # variable bar for everything
                plt.scatter(p[iv,0], p[iv,1], c=ep, s=ms, vmin=0, vmax=np.max(ep) if n in [IY, IYD] else 1.0) # fixed bar but not for Iy
                prep_plot(plt.gca()), plt.colorbar()

                if on_grid: # plot contours if on grid
                    plt.subplot(nraws, ncols, j*(ncols//2) + 3), plt.title(f"{n} Contours")
                    plt.contour(RRD, ZZD, v.reshape(65,28), levels=20, linestyles='solid', cmap=cm)
                    plt.contour(RRD, ZZD, vp.reshape(65,28), levels=20, linestyles='dashed', cmap=cm)
                    prep_plot(plt.gca()), plt.colorbar()
            fig = plt.gcf() # Get the current figure
            dx, xoff = 0.004, 0.01 #0.005, 0.008
            # dy, yoff = dx*fs[0]/fs[1], -0.012 #dx*fs[0]/fs[1], 0.003
            dy, yoff = dx*fs[0]/fs[1], -0.018 #dx*fs[0]/fs[1], 0.003
            tvs = 0.041 # title verticale space
            from matplotlib.patches import Rectangle
            # for x0, y0, lx, ly in [(0.0+dx, 0.5+dy/2+yoff, (1-3*dx)/2+xoff, (1-3*dy)/2-yoff-tvs),
            #                        (0.5+dx/2+xoff, 0.5+dy/2+yoff, (1-3*dx)/2-xoff, (1-3*dy)/2-yoff-tvs),
            #                        (0.0+dx, 0.0+dy, (1-3*dx)/2+xoff, (1-3*dy)/2+yoff),
            #                        (0.5+dx/2+xoff, 0.0+dy, (1-3*dx)/2-xoff, (1-3*dy)/2+yoff)]:
            for x0, y0, lx, ly in [(0.0+dx,         0.5+dy/2+yoff, (1-3*dx)/2+xoff, (1-3*dy)/2-yoff-tvs),
                                (0.5+dx/2+xoff,  0.5+dy/2+yoff, (1-3*dx)/2-xoff, (1-3*dy)/2-yoff-tvs),
                                (0.0+dx,         0.0+dy, (1-3*dx)/2+xoff, (1-3*dy)/2+yoff),
                                (0.5+dx/2+xoff,  0.0+dy, (1-3*dx)/2-xoff, (1-3*dy)/2+yoff)]:
                fig.patches.append(Rectangle( (x0, y0), lx, ly, transform=fig.transFigure, color='black' if DARK_THEME else 'white', alpha=1, zorder=-1))
            plt.suptitle(f"[{JOBID}] liuqeML: {title} {i}", y=0.985, color='black' if DARK_THEME else 'white')
            plt.tight_layout()
            plt.savefig(f"{save_dir}/imgs/net_example_{'g_' if on_grid else ''}{title}_{i}.png")
            if LOCAL: plt.show()  
            plt.close()

    ds.pps = _ds_pps # restore dataset pps
    return


In [ ]:
# testing network outputs
# try:
if True:
    print("Testing network output...")
    for active, ln in zip(ACTIVE_LOSSES, LOSS_NAMES):
        if not active: continue
        model = liuqeML(train_ds.x_mean_std)
        model.load_state_dict(torch.load(model_path(ln), map_location=torch.device(CPU))) # load pretrained model
        use_db = R_BD[-1] > R_B[-1] or (R_BD[-1] == 0 and R_B[-1] == 0)
        use_diy = R_IYD[-1] > R_IY[-1] or (R_IYD[-1] == 0 and R_IY[-1] == 0)
        plot_network_outputs(val_ds, model, title=ln, use_db=use_db, use_diy=use_diy) # plot network outputs
# except Exception as e:
#     print(f"Error in testing network output: {e}")
#     print("Skipping network output testing...")
#     pass

In [ ]:
# save weights to .mat file
m = liuqeML(train_ds.x_mean_std)
for active, ln in zip(ACTIVE_LOSSES, LOSS_NAMES):
    if active:
        m.load_state_dict(torch.load(model_path(ln), map_location=torch.device(CPU))) # load pretrained model
        save_weights_to_mat(m, title=ln)

In [ ]:
print(f"{JOBID} done")
if not LOCAL: sleep(30) # wait for files to update (for cluster)
#copy the log file to the folder
os.system(f"cp jobs/{JOBID}.txt {SAVE_DIR}/log.txt")